# فاز دوم – معرفی ویژگی‌های دیتاست

در این فاز، متغیرهای مورد استفاده در تحلیل‌های آماری و احتمالاتی دیتاست NHANES معرفی می‌شوند.

| ویژگی | توضیحات |
|---|---|
| `SEQN` | شناسه منحصربه‌فرد هر شرکت‌کننده در مطالعه |
| `INDFMPIR` | نسبت درآمد خانواده به سطح فقر (Income-to-Poverty Ratio)؛ مقادیر بیشتر از ۵ نیز برابر ۵ ثبت شده‌اند. |
| `WTINT2YR` | وزن نمونه‌گیری مصاحبه برای دوره دوساله |
| `RIDAGEYR` | سن شرکت‌کننده هنگام غربالگری؛ سن افراد ۸۰ سال و بالاتر برابر ۸۰ ثبت شده است. |
| `RIAGENDR` | جنسیت شرکت‌کننده |
| `DMDEDUC2` | بالاترین سطح تحصیلات افراد ۲۰ سال به بالا |
| `INQ300` | وضعیت داشتن بیش از ۵۰۰۰ دلار پس‌انداز توسط خانواده (۱: بله، ۲: خیر) |
| `SLQ300` | ساعت معمول به خواب رفتن در روزهای کاری یا روزهای هفته |
| `SLQ310` | ساعت معمول بیدار شدن در روزهای کاری یا روزهای هفته |
| `SLQ320` | ساعت معمول به خواب رفتن در آخر هفته‌ها یا روزهای غیرکاری |
| `SLQ330` | ساعت معمول بیدار شدن در آخر هفته‌ها یا روزهای غیرکاری |

## وزن نمونه‌گیری (Survey Weight)

داده‌های NHANES با استفاده از روش نمونه‌گیری احتمالی چندمرحله‌ای جمع‌آوری شده‌اند. بنابراین، هر شرکت‌کننده نماینده تعداد مشخصی از افراد جامعه آماری است.

متغیر `WTINT2YR` وزن نمونه‌گیری مصاحبه برای یک دوره دوساله است و نشان می‌دهد هر مشاهده چه سهمی از جمعیت هدف را نمایندگی می‌کند.

### محاسبه میانگین وزنی

برای محاسبه میانگین وزنی یک متغیر، از رابطه زیر استفاده می‌شود:
$$
\bar{X}_w = \frac{\sum_{i=1}^{n} w_i x_i}{\sum_{i=1}^{n} w_i}
$$

که در آن:

- \(x_i\): مقدار متغیر مورد بررسی برای فرد \(i\)
- \(w_i\): وزن نمونه‌گیری فرد، یعنی مقدار `WTINT2YR`
- \(n\): تعداد مشاهدات معتبر

In [1]:
%run ../src/data_summary.py


========== GENERAL STATISTICS ==========

Weekday_Sleep
Mean: 7.77
Median: 8.0
Minimum: 3.0
Maximum: 14.0
Standard Deviation: 1.47

Weekend_Sleep
Mean: 8.62
Median: 8.5
Minimum: 3.0
Maximum: 14.0
Standard Deviation: 1.62

========== SLEEP BY GENDER ==========
          Weekday_Sleep  Weekend_Sleep
RIAGENDR                              
Female             7.88           8.71
Male               7.63           8.51

========== SLEEP BY AGE GROUP ==========
           Participants  Weekday_Mean  Weekend_Mean
Age_Group                                          
20-22               234          8.20          9.01
23-25               259          7.83          8.76
26-28               285          7.76          8.60
29-31               349          7.62          8.50
32-35               482          7.63          8.46

========== AGE FREQUENCY ==========
RIDAGEYR
20.0     70
21.0     81
22.0     83
23.0     82
24.0     83
25.0     94
26.0     84
27.0    105
28.0     96
29.0    113
30.0    117

# وقتی آخر هفته می‌رسه، خوابمون رو پس می‌گیریم؟

این بخش قراره با زبان احتمال، دو تا رفتار جالب مرتبط با خواب رو بررسی کنه.

## ایده اول: جبران خواب آخر هفته (Weekend Sleep Compensation)

فرضیه ساده‌ست: بعضی از ما در طول هفته به‌اندازه کافی نمی‌خوابیم و سعی می‌کنیم آخر هفته این کمبود رو جبران کنیم.

برای بررسی این موضوع دو متغیر تصادفی تعریف می‌کنیم:

- \(X\): مدت خواب در روزهای کاری
- \(Y\): مدت خواب در آخر هفته

حالا دو رویداد تعریف می‌کنیم:

- \(A\): خواب کمتر از 6٫۷۷ ساعت در روزهای کاری
- \(B\): خواب بیشتر از 9٫۶۲ ساعت در آخر هفته

این آستانه‌ها با فاصله یک ساعته از میانگین خواب نمونه انتخاب شده‌اند.

چیزی که دنبالش هستیم احتمال وقوع \(B\) به شرط وقوع \(A\) است:

$$
P(B\mid A)=\frac{P(A\cap B)}{P(A)}
$$

به زبان ساده: از بین افرادی که در طول هفته کم خوابیده‌اند، چند درصد در آخر هفته خواب بیشتری داشته‌اند؟

## ایده دوم: جغدهای شب (Night Owls)

همه آدم‌ها ساعت زیستی یکسانی ندارند. بعضی‌ها زودتر می‌خوابند و زودتر بیدار می‌شوند، بعضی‌ها هم ترجیح می‌دهند شب را طولانی‌تر کنند!

برای بررسی این تفاوت، به‌جای صرفاً مدت خواب، سراغ نقطه میانی خواب در آخر هفته می‌رویم.

متغیر تصادفی \(Z\) را به‌صورت ساعت نقطه میانی فاصله خوابیدن تا بیدار شدن تعریف می‌کنیم:

$$
Z=(T_s+\frac{D}{2})\bmod 24
$$

در این رابطه، \(T_s\) ساعت خوابیدن و \(D\) مدت خواب است.

به‌صورت عملیاتی، افرادی را که نقطه میانی خوابشان ساعت 5 بامداد یا دیرتر باشد، در گروه Night Owls قرار می‌دهیم:

$$
O=\{Z\geq 3\}
$$

در نهایت احتمال مشاهده این الگو را در نمونه محاسبه می‌کنیم:

$$
P(O)=\frac{N(O)}{N}
$$

این تعریف قرار نیست Chronotype افراد را به‌طور قطعی تشخیص دهد؛ فقط یک معیار ساده برای بررسی الگوی دیرخوابی در داده‌های موجود است.